# V0.5

In [ ]:
!pip install -q kaggle-environments

In [ ]:
%%writefile main.py
import random

def step_toward(fx, fy, tx, ty, tiles):
    """
    最短ルートを計算
    目的地(tx, ty)に向かって安全に1歩進む

    1_X軸（東西）のチェック 目的地が右にあり、マップ右端(9)ではなく、右マスがLOCKEDでない場合：
      EASTを追加
    2_左（WEST）のチェック
    3_Y軸（南北）のチェック 目的地が下にあり、マップ下端(9)ではなく、下マスがLOCKEDでない場合：
      SOUTHを追加
    4_上（NORTH）のチェック
    5_目的地に近づく安全なルートがある場合：
      その中からランダムに1歩選ぶ（斜め移動時の角への引っかかりを防ぐため）
    6_目的地に近づけない場合：
     （障害物に塞がれている等）は、とりあえず移動可能な安全なマスを探す
    7_動ける場所があればランダム移動、完全に閉じ込められていればPASSを返す
    """
    candidates = []


    if fx < tx and fx < 9 and tiles[fy][fx + 1] != "LOCKED":
        candidates.append("EAST")
    elif fx > tx and fx > 0 and tiles[fy][fx - 1] != "LOCKED":
        candidates.append("WEST")
    if fy < ty and fy < 9 and tiles[fy + 1][fx] != "LOCKED":
        candidates.append("SOUTH")
    elif fy > ty and fy > 0 and tiles[fy - 1][fx] != "LOCKED":
        candidates.append("NORTH")
    if candidates:
        return random.choice(candidates)

    valid_dirs = []
    if fy > 0 and tiles[fy - 1][fx] != "LOCKED": valid_dirs.append("NORTH")
    if fy < 9 and tiles[fy + 1][fx] != "LOCKED": valid_dirs.append("SOUTH")
    if fx < 9 and tiles[fy][fx + 1] != "LOCKED": valid_dirs.append("EAST")
    if fx > 0 and tiles[fy][fx - 1] != "LOCKED": valid_dirs.append("WEST")

    return random.choice(valid_dirs) if valid_dirs else "PASS"

def find_target_tile(tiles, fx, fy, have_seeds, day):
    """
    一番優先度の高い仕事場(X, Y)を探す
    畑全体スキャン

    1_10x10のマップ全体をループで確認
    2_未解放エリアは作業対象外なのでスキップ
    3_対象のマスに植物(PLANT)がある場合:
      植えてから2日以上経過していれば「収穫候補」として追加
      その日の水やりが終わっていなければ「水やり候補」として追加

    4_対象のマスが空き地(None)で、かつ種を持っている場合:
      種まき候補として追加

    5_畑に何も仕事がない場合:Noneを返す
    """
    candidates = []

    for y in range(len(tiles)):

        for x in range(len(tiles[0])):
            t = tiles[y][x]
            if t == "LOCKED": continue
            if isinstance(t, dict) and t.get("kind") == "PLANT":
                crop_age = day - t.get("planted_day", day)
                if crop_age >= 2:
                    candidates.append((x, y, "harvest"))
                elif not t.get("watered_today", True):
                    candidates.append((x, y, "water"))
            elif t is None and have_seeds:
                candidates.append((x, y, "plant"))
    if not candidates:
        return None

    # 仕事の優先順位を定義（数字が小さいほど最優先。収穫 > 水やり > 種まき）
    priority = {"harvest": 0, "water": 1, "plant": 2}

    # 候補リストを並び替える
    # 基準1: 上記の優先順位（最重要）
    # 基準2: 現在地(fx, fy)からのマンハッタン距離（近いものを優先）
    candidates.sort(key=lambda c: (priority[c[2]], abs(c[0] - fx) + abs(c[1] - fy)))

    # 一番条件の良い仕事場の座標(X, Y)を返す
    return candidates[0][0], candidates[0][1]



def agent(obs, config):
    player = obs["player"]
    me = obs["farms"][player]
    private = obs["private"]

    fx, fy = me["farmer"]
    tiles = me["tiles"]
    tile = tiles[fy][fx]

    # 辞書にキーが無い場合のエラーを防ぐ
    money = me.get("money", 0)
    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    day = obs.get("day", 0)

    # 現在の労働者のリストを取得
    current_hands = me.get("hands", [])

    market = []

    # ========================================
    # 1. 市場での売買・雇用ロジック
    # ========================================
    wheat_seeds = seeds.get("WHEAT", 0)

    # 種がなく、お金が10以上あれば小麦の種を買う
    if wheat_seeds == 0 and money >= 10:
        market.append(["BUY_SEED", "WHEAT", 1])

    # 小屋に小麦があればすべて売る
    wheat_in_shed = shed.get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", wheat_in_shed])

    #労働者が0人で、お金が十分にある時だけ1人雇う
    if money >= 100 and len(current_hands) == 0:
        market.append(["HIRE"])

    # ========================================
    # 2. メイン農家のアクション（最優先）
    # ========================================
    farmer_action = None

    # 足元が空き地で、種を持っているなら種まき
    if tile is None and wheat_seeds > 0:
        farmer_action = ["PLANT", "WHEAT"]

    # 足元に植物がある場合：
    elif isinstance(tile, dict) and tile.get("kind") == "PLANT":
        crop_age = day - tile.get("planted_day", day)

        # 2日経過で収穫
        if crop_age >= 2:
            farmer_action = ["HARVEST"]

        # 今日の水やりがまだなら水やり
        elif not tile.get("watered_today", True):
            farmer_action = ["WATER"]

    # ========================================
    # 3. 農家の移動
    # ========================================

    # 足元でやることがなかった場合、レーダーを使って次の目的地を探す
    if farmer_action is None:
        target = find_target_tile(tiles, fx, fy, wheat_seeds > 0, day)

        if target:
            # 目的地が見つかれば、ナビを使って1歩進む
            move_dir = step_toward(fx, fy, target[0], target[1], tiles)
            farmer_action = [move_dir]
        else:
            # 畑全体に仕事がなければ待機
            farmer_action = ["PASS"]

    # ========================================
    # 4. 労働者(hands)の行動を決定する
    # ========================================
    hands_actions = []

    # 雇っている人数分だけループを回して、個別に指示を出す
    for hand in current_hands:
        hx, hy = hand
        hand_tile = tiles[hy][hx]
        hand_action = None

        # 労働者は水やりと収穫を手伝う
        if isinstance(hand_tile, dict) and hand_tile.get("kind") == "PLANT":
            crop_age = day - hand_tile.get("planted_day", day)
            if crop_age >= 2:
                hand_action = ["HARVEST"]
            elif not hand_tile.get("watered_today", True):
                hand_action = ["WATER"]

        # 農家と同じように一番近い仕事場へ向かって進む
        if hand_action is None:
            target = find_target_tile(tiles, hx, hy, False, day)
            if target:
                move_dir = step_toward(hx, hy, target[0], target[1], tiles)
                hand_action = [move_dir]
            else:
                hand_action = ["PASS"]

        hands_actions.append(hand_action)

    # ========================================
    # 5. 最終的な行動をまとめて返す
    # ========================================
    return {
        "farmer": farmer_action,
        "hands": hands_actions,
        "market": market
    }

Overwriting main.py


In [ ]:
from zoneinfo import ZoneInfo
import datetime
from kaggle_environments import make
from IPython.display import HTML



print(datetime.datetime.now(ZoneInfo("Asia/Tokyo")))


env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)

env.run(["main.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

html_output = env.render(mode="html", width=800, height=600)
HTML(html_output)


Output hidden; open in https://colab.research.google.com to view.